# Снижение добавленной сахарозы в песочном печенье

## 1. Задача, производство и данные

**Бизнес-вопрос:** насколько можно сократить количество добавленной сахарозы в рецептуре песочного печенья с помощью пищевых волокон и какие изменения качества при этом возникают?

Кондитерское производство хочет уменьшить добавление сахарозы и сохранить характеристики привычного продукта максимально близкими к стандарту. Мы сравниваем только четыре опубликованных варианта со стандартным печеньем. Они отличаются и ингредиентом, и его дозировкой. Такой эксперимент не определяет абсолютный предел снижения сахарозы и не позволяет построить универсальную зависимость качества от процента замены.

**Статус:** этап 2 — подготовка и проверка таблиц. Здесь выполнены загрузка, преобразование и описательные расчёты для будущего анализа. Статистические тесты, доверительные интервалы, финальные графики, дашборд и выбор рецептуры ещё не выполнены.

### Что мы меняем в рецептуре и зачем?

Сахароза — обычный столовый сахар. В печенье она отвечает не только за сладость: взаимодействует с водой, влияет на свойства теста, его растекание при выпечке, цвет и структуру готового изделия. Поэтому просто убрать часть сахара и сохранить прежний результат трудно.

Пищевые волокна — компоненты пищевого сырья, которые не перевариваются ферментами тонкого кишечника человека. В этом исследовании используют растворимые волокна: они могут взаимодействовать с водой в тесте и частично заменять сахар по массе. Но они отличаются от сахарозы строением и поведением в рецептуре. Растворимость не означает одинаковую сладость или одинаковое влияние на структуру.

В эксперименте две группы ингредиентов:

- **Резистентные декстрины** получают переработкой крахмала. Nutriose® FM06 и Promitor® SGF 70R — два разных коммерческих ингредиента из кукурузного сырья. Мы рассматриваем их отдельно.
- **Инулин** — пищевое волокно из цепочек фруктозных звеньев. Orafti® HSI и Fibruline™ Instant могут различаться распределением длины цепей, растворимостью и технологическим поведением. Название «Длинноцепочечный инулин» здесь обозначает Fibruline из задания; это смесь цепей, а не только длинные молекулы.

Бытовой пример: если из привычного рецепта печенья убрать часть сахара и заменить его другим веществом, печенье может стать менее сладким, более плотным, изменить цвет или сильнее крошиться. Это возможные последствия замены, а не заранее установленные результаты для конкретного ингредиента. Замена волокнами не гарантирует сохранение вкуса и текстуры и сама по себе не доказывает, что продукт полезнее или менее калориен.

### Источники и пять рецептур

Используем уже загруженные оригиналы из `source/`, [отчёт первого этапа](audit_report.md) и [атрибуцию источников](DATA_LICENSE.md).

- [Набор Университета Рединга, DOI 10.17864/1947.000384](https://doi.org/10.17864/1947.000384).
- [Rodriguez-Garcia et al. (2022), DOI 10.1016/j.lwt.2022.113837](https://doi.org/10.1016/j.lwt.2022.113837): состав в таблице 2, измерения в разделе 2; определения сенсорных признаков — в дополнительной таблице S1.

**«Г на 100 г муки»** — рецептурное соотношение: при использовании 100 г муки берут указанную массу ингредиента. Это не процент сахарозы в готовом печенье. Снижение добавленной сахарозы также не равно измеренному снижению всех сахаров готового продукта: такого анализа в наборе нет.

В таблице ниже «волокна» означают массу добавленного коммерческого ингредиента с волокнами. Это не измеренная масса чистых пищевых волокон: продукты содержат и другие компоненты (таблица 1 статьи). Пересчитывать её в «чистые волокна» по неподтверждённой точной доле не будем.

In [1]:
from pathlib import Path
import hashlib
import importlib.metadata
import json
import re
import sys

import numpy as np
import pandas as pd
import pdfplumber
from docx import Document
from openpyxl import load_workbook
from openpyxl.utils import get_column_letter
from openpyxl.utils.cell import column_index_from_string
from IPython.display import display

ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / 'source' / 'BiscuitData.xlsx').exists() and (p / 'audit_report.md').exists()),
    None,
)
if ROOT is None:
    raise FileNotFoundError('Запустите notebook из корня репозитория cookies.')
SOURCE = ROOT / 'source'
DATA = ROOT / 'data'
DATA.mkdir(exist_ok=True)

manifest = json.loads((SOURCE / 'download_manifest.json').read_text(encoding='utf-8'))
source_hashes_before = {}
for entry in manifest:
    digest = hashlib.sha256((SOURCE / entry['file']).read_bytes()).hexdigest()
    assert digest == entry['sha256'], f"Изменился исходник: {entry['file']}. Нужен повторный аудит."
    source_hashes_before[entry['file']] = digest

recipe_meta = pd.DataFrame([
    ('control', 'Control', 'Стандартная', '', 'Стандарт', 0),
    ('nutriose', 'Nutriose', 'Кукурузный декстрин № 1', 'Nutriose® FM06', 'Резистентный декстрин', 1),
    ('promitor', 'Promitor', 'Кукурузный декстрин № 2', 'Promitor® SGF 70R', 'Резистентный декстрин', 2),
    ('orafti', 'Orafti', 'Более растворимый инулин', 'Orafti® HSI', 'Инулин', 3),
    ('fibruline', 'Fibruline', 'Длинноцепочечный инулин', 'Fibruline™ Instant', 'Инулин', 4),
], columns=['recipe_id', 'recipe_original', 'recipe_name_ru', 'ingredient_original', 'ingredient_group_ru', 'recipe_order'])
recipe_order = recipe_meta['recipe_id'].tolist()
recipe_original_order = recipe_meta['recipe_original'].tolist()
original_to_id = dict(zip(recipe_meta['recipe_original'], recipe_meta['recipe_id']))

versions = {'Python': sys.version.split()[0]}
for package in ['pandas', 'numpy', 'openpyxl', 'pdfplumber', 'python-docx']:
    versions[package] = importlib.metadata.version(package)
display(pd.DataFrame(versions.items(), columns=['Компонент', 'Версия']))
print('Контрольные суммы всех шести исходных документов совпадают с аудитом.')

,Компонент,Версия
0,Python,3.12.14
1,pandas,2.2.3
2,numpy,2.3.5
3,openpyxl,3.1.5
4,pdfplumber,0.11.9
5,python-docx,1.2.0


Контрольные суммы всех шести исходных документов совпадают с аудитом.


In [2]:
# Состав извлекается из самой таблицы 2 PDF, а не из вручную заполненной CSV.
with pdfplumber.open(SOURCE / 'article.pdf') as pdf:
    page = pdf.pages[4]
    left_text = page.crop((0, 0, page.width / 2, page.height)).extract_text()
lines = left_text.splitlines()
start = lines.index('Ingredients Control Nutriose Promitor Orafti Fibruline') + 1
end = next(i for i in range(start, len(lines)) if lines[i].startswith('TS '))

ingredient_prefixes = {
    'Flour': 'flour', 'Butter': 'butter', 'Sucrose': 'sucrose',
    'Nutriose': 'nutriose_ingredient', 'Promitor': 'promitor_ingredient',
    'Orafti': 'orafti_ingredient', 'Fibruline': 'fibruline_ingredient',
    'Skimmed milk powder': 'skimmed_milk_powder', 'Water': 'water', 'Salt': 'salt',
    'Sodium bicarbonate': 'sodium_bicarbonate', 'Ammonium hydrogen': 'ammonium_bicarbonate',
}
recipe_columns = {}
for line in lines[start:end]:
    prefix = next((key for key in ingredient_prefixes if line.startswith(key + ' ') or line.startswith(key + '®') or line.startswith(key + '™')), None)
    if prefix is None:
        continue
    values = [float(token) for token in line.split()[-5:]]
    column = ingredient_prefixes[prefix] + '_g_per_100g_flour'
    assert column not in recipe_columns, f'Повтор строки состава: {column}'
    recipe_columns[column] = values
assert len(recipe_columns) == 12

recipes = pd.concat([recipe_meta, pd.DataFrame(recipe_columns)], axis=1)
mass_columns = list(recipe_columns)
fibre_columns = [c for c in mass_columns if '_ingredient_' in c]
recipes['fibre_ingredient_g_per_100g_flour'] = recipes[fibre_columns].sum(axis=1)
control_sucrose = recipes.loc[recipes['recipe_id'].eq('control'), 'sucrose_g_per_100g_flour'].iloc[0]
recipes['sucrose_removed_g_per_100g_flour'] = control_sucrose - recipes['sucrose_g_per_100g_flour']
recipes['sucrose_reduction_pct'] = 100 * recipes['sucrose_removed_g_per_100g_flour'] / control_sucrose
recipes['total_recipe_g_per_100g_flour'] = recipes[mass_columns].sum(axis=1)
control_total = recipes.loc[recipes['recipe_id'].eq('control'), 'total_recipe_g_per_100g_flour'].iloc[0]
recipes['total_recipe_delta_vs_control_g'] = recipes['total_recipe_g_per_100g_flour'] - control_total
recipes['claimed_reduction_pct'] = [0, 30, 30, 30, 30]
recipes['claim_matches_recipe_table'] = np.isclose(recipes['claimed_reduction_pct'], recipes['sucrose_reduction_pct'], atol=0.005)
recipes['total_sugars_measured'] = False
recipes['source_file'] = 'article.pdf'
recipes['source_table'] = 'Table 2'
recipes['source_pdf_page'] = 5
recipes['source_article_page'] = 3
recipes['qc_flags'] = np.where(recipes['recipe_id'].eq('control'), '', 'claimed_30_percent_differs_from_recipe_table')
recipes.loc[recipes['recipe_id'].eq('orafti'), 'qc_flags'] += ';orafti_total_recipe_mass_differs'

assert np.allclose(recipes['sucrose_g_per_100g_flour'], [29.45, 20.18, 19.11, 22.94, 19.19])
assert np.allclose(recipes['fibre_ingredient_g_per_100g_flour'], [0, 9.27, 10.34, 10.72, 10.26])
assert np.allclose(recipes['sucrose_reduction_pct'].round(2), [0, 31.48, 35.11, 22.11, 34.84])
assert np.allclose(recipes['total_recipe_g_per_100g_flour'], [175.95, 175.95, 175.95, 180.16, 175.95])

display(recipes[['recipe_name_ru', 'recipe_original', 'sucrose_g_per_100g_flour',
                 'fibre_ingredient_g_per_100g_flour', 'sucrose_reduction_pct',
                 'total_recipe_g_per_100g_flour']].rename(columns={
    'recipe_name_ru': 'Рецептура', 'recipe_original': 'Обозначение авторов',
    'sucrose_g_per_100g_flour': 'Сахароза, г/100 г муки',
    'fibre_ingredient_g_per_100g_flour': 'Волокна (масса ингредиента), г/100 г муки',
    'sucrose_reduction_pct': 'Снижение добавленной сахарозы, %',
    'total_recipe_g_per_100g_flour': 'Сумма ингредиентов, г/100 г муки',
}).round(2))

,Рецептура,Обозначение авторов,"Сахароза, г/100 г муки","Волокна (масса ингредиента), г/100 г муки","Снижение добавленной сахарозы, %","Сумма ингредиентов, г/100 г муки"
0,Стандартная,Control,29.45,0.00,0.00,175.95
1,Кукурузный декстрин № 1,Nutriose,20.18,9.27,31.48,175.95
2,Кукурузный декстрин № 2,Promitor,19.11,10.34,35.11,175.95
3,Более растворимый инулин,Orafti,22.94,10.72,22.11,180.16
4,Длинноцепочечный инулин,Fibruline,19.19,10.26,34.84,175.95


Вариант Orafti содержит на 4,21 г больше суммарной массы ингредиентов на 100 г муки: удалено 6,51 г сахарозы, добавлено 10,72 г заменителя. Сохраняем опубликованный состав и отмечаем расхождение. Причина неизвестна; исправлять его предположением или пересчитывать все рецептуры к одинаковой массе нельзя.

Указанные в тексте статьи «30%» — заявленная характеристика эксперимента. Для нашего бизнес-вопроса используем расчёт по реальным строкам рецептурной таблицы, без округления всех вариантов до 30%.

## 2. Подготовка и проверка данных

### Уровни наблюдений

По статье изготовлено три партии каждой рецептуры в разные дни. Блоки `Replicate 1–3` в физических таблицах используем как **условные обозначения партий**. Дат и лабораторных batch_id нет. Одинаковый номер блока у разных рецептур не доказывает общий день и не создаёт парность.

| Измерение | Наблюдения внутри партии | Независимая единица будущего сравнения |
|---|---|---|
| Проникновение и число пиков | 10 изделий | Производственная партия |
| Излом: сила и перемещение | 10/10/9 изделий в трёх партиях | Производственная партия |
| Цвет | 3 изделия | Производственная партия |
| Влажность и aw | 3 технических повтора смеси крошки 10 изделий | Производственная партия |
| Масса до/после выпечки, потери массы | Групповой результат по 10 изделиям | Производственная партия |
| Размеры | Один групповой результат по 10 изделиям | Производственная партия |
| Сенсорика | Только опубликованные средние, без индивидуальных строк | Новые тесты недоступны |

Итого для каждого физического показателя — три производственных повтора на рецептуру. Мы сохраняем все доступные измерения, но не выдаём отдельные печенья за дополнительные независимые партии.

### Загрузка книг и единицы

Pandas читает все листы трёх книг. Openpyxl дополнительно читает исходные формулы и подписи, не пересчитывая и не изменяя Excel. Состав берём из PDF. Для физических измерений основной источник — `BiscuitData.xlsx`.

Размеры используем в **мм**: это подтверждают заголовки измерений и таблица 4 статьи; указание сантиметров в README/Index сохраняем как конфликт документации. Силу разрушения используем в **Н**: её подтверждают первый блок, Index и раздел 2.5.4 статьи. Ошибочные подписи `N.s` в следующих блоках не меняют исходные числа и остаются в полях происхождения и флагах.

Сопротивление проникновению в наборе — площадь кривой в **Н·с**, а не максимальная сила в Н. «Хрупкость» (Fracturability) представлена перемещением зонда в момент излома, **мм**; отдельной измеренной шкалы хрупкости нет. Цветовые координаты не приводим к общей шкале с силой или размерами.

In [3]:
books = {}
inventory = []
for filename in ['IngredientData.xlsx', 'DoughData.xlsx', 'BiscuitData.xlsx']:
    with pd.ExcelFile(SOURCE / filename, engine='openpyxl') as book:
        books[filename] = {}
        for sheet in book.sheet_names:
            frame = pd.read_excel(book, sheet_name=sheet, header=None, dtype=object)
            books[filename][sheet] = frame
            inventory.append((filename, sheet, len(frame), len(frame.columns)))
display(pd.DataFrame(inventory, columns=['Файл', 'Лист', 'Строк в таблице', 'Столбцов']))
assert len(inventory) == 17

formula_book = load_workbook(SOURCE / 'BiscuitData.xlsx', data_only=False, read_only=True)
biscuit_sheets = books['BiscuitData.xlsx']

# Код, русское имя, единица, пояснение, роль, относительное отклонение допустимо.
physical_specs = [
    ('breaking_strength_n', 'Сила разрушения', 'Н', 'Максимальная сила при трёхточечном изломе печенья. Не равна оценке твёрдости укуса дегустатором.', 'primary', True),
    ('penetration_resistance_ns', 'Сопротивление проникновению', 'Н·с', 'Площадь кривой усилия при проникновении зонда в печенье.', 'secondary', True),
    ('number_of_peaks', 'Число пиков', 'шт.', 'Количество локальных пиков в испытании на проникновение; характеризует локальные разрушения структуры.', 'secondary', True),
    ('deformation_at_break_mm', 'Перемещение при разрушении', 'мм', 'Перемещение зонда в момент излома; в статье обозначено Fracturability.', 'secondary', True),
    ('height_mm', 'Высота', 'мм', 'Высота изделия по групповому измерению партии.', 'secondary', True),
    ('width_mm', 'Ширина', 'мм', 'Ширина изделия по групповому измерению партии.', 'secondary', True),
    ('length_mm', 'Длина', 'мм', 'Длина изделия по групповому измерению партии.', 'secondary', True),
    ('moisture_pct', 'Влажность', '%', 'Содержание воды, измеренное высушиванием смеси крошки до постоянной массы.', 'secondary', True),
    ('water_activity', 'Активность воды', 'безразм.', 'Показатель доступности воды для реакций и роста микроорганизмов. Сам по себе не устанавливает срок хранения.', 'secondary', False),
    ('weight_loss_pct', 'Потеря массы при выпечке', '%', 'Процент уменьшения массы группы изделий: (масса до − масса после) / масса до × 100.', 'secondary', True),
    ('colour_l_star', 'Светлота L*', 'ед. CIELAB', 'Цветовая координата от чёрного (0) к белому (100).', 'secondary', False),
    ('colour_a_star', 'Цветовая координата a*', 'ед. CIELAB', 'Отрицательные значения связаны с зелёным, положительные — с красным направлением цвета.', 'secondary', False),
    ('colour_b_star', 'Цветовая координата b*', 'ед. CIELAB', 'Отрицательные значения связаны с синим, положительные — с жёлтым направлением цвета.', 'secondary', False),
    ('weight_before_g', 'Масса группы до выпечки', 'г', 'Масса десяти заготовок. Контекст расчёта потерь, а не масса одного печенья.', 'context', True),
    ('weight_after_g', 'Масса группы после выпечки', 'г', 'Масса группы десяти изделий после выпечки. Не масса одного изделия.', 'context', True),
]
physical_dictionary = pd.DataFrame(physical_specs, columns=[
    'metric_id', 'metric_name_ru', 'unit', 'definition_ru', 'metric_role', 'relative_delta_allowed'])
physical_dictionary['domain'] = 'physical'
physical_dictionary['section_ru'] = 'Физические и текстурные измерения'
physical_dictionary['metric_label_ru'] = physical_dictionary['metric_name_ru']
physical_dictionary['delta_unit'] = physical_dictionary['unit'].replace({'%': 'п.п.'})
physical_dictionary['statistical_unit'] = 'production_batch'
physical_dictionary['new_statistical_tests_allowed'] = True
physical_dictionary['definition_source_file'] = 'article.pdf;README-SureChoc.docx'
physical_dictionary['definition_source_location'] = 'Methods 2.5; README variables'
physical_dictionary['definition_status'] = 'supported_by_methods'
physical_dictionary['metric_original'] = [
    'Breaking strength (N)', 'Resistance to penetration (N.s)', 'Number of peaks',
    'Fracturability (mm)', 'Height (mm)', 'Width (mm)', 'Length (mm)',
    'Moisture (%)', 'Water activity (aw)', '% Loss', 'L*', 'a*', 'b*',
    'Weight Before (g)', 'Weight After (g)',
]
metric_units = physical_dictionary.set_index('metric_id')['unit'].to_dict()

,Файл,Лист,Строк в таблице,Столбцов
0,IngredientData.xlsx,Index,10,3
1,IngredientData.xlsx,Table 1_DP,9,27
2,DoughData.xlsx,Index,14,3
3,DoughData.xlsx,Figure 1_viscoelastic prop,143,27
4,DoughData.xlsx,Figure 2_Tan Loss,21,6
5,BiscuitData.xlsx,Index,29,3
6,BiscuitData.xlsx,Table 3_WL,4,20
7,BiscuitData.xlsx,Table 3_aw,4,23
8,BiscuitData.xlsx,Table 3_Moisture,4,23
9,BiscuitData.xlsx,Table 4_Dimensions,4,20


In [4]:
records = []
missing_records = []

def cell_name(row, col):
    return f'{get_column_letter(col + 1)}{row + 1}'

def source_value(sheet, row, col):
    frame = biscuit_sheets[sheet]
    return frame.iat[row, col] if row < len(frame) and col < len(frame.columns) else np.nan

def add_block(sheet, metric, value_rows, starts, header_row, label_row,
              assay, level, expected_values, n_group_items=None, flags=''):
    for batch_number, start_col in enumerate(starts, 1):
        assert source_value(sheet, 0, start_col - (2 if level == 'technical_repeat' else 1)) == f'Replicate {batch_number}'
        for offset, original in enumerate(recipe_original_order):
            col = start_col + offset
            assert source_value(sheet, header_row, col) == original
            recipe_id = original_to_id[original]
            batch_id = f'{recipe_id}_R{batch_number}'
            metric_header_col = start_col - 1
            for position, row in enumerate(value_rows, 1):
                address = cell_name(row, col)
                value = source_value(sheet, row, col)
                if pd.isna(value):
                    missing_records.append((recipe_id, batch_id, metric, address))
                    continue
                if not isinstance(value, (int, float, np.number)) or not np.isfinite(value):
                    raise ValueError(f'Нечисловое измерение: {sheet}!{address}: {value}')
                has_local_label = level == 'technical_repeat' or (level == 'item' and assay != 'colour')
                original_label = source_value(sheet, row, start_col - 2 if level == 'technical_repeat' else start_col - 1) if has_local_label else pd.NA
                original_header = source_value(sheet, label_row, metric_header_col)
                formula = formula_book[sheet][address].value
                formula = formula if isinstance(formula, str) and formula.startswith('=') else ''
                qc = flags
                if metric == 'breaking_strength_n' and batch_number > 1:
                    qc = ';'.join(filter(None, [qc, 'breaking_header_unit_conflict_resolved_by_methods']))
                records.append({
                    'measurement_id': f'BiscuitData.xlsx|{sheet}|{address}',
                    'recipe_id': recipe_id, 'batch_id': batch_id,
                    'source_replicate': batch_number, 'actual_batch_id': pd.NA,
                    'experiment_date': pd.NA, 'shared_day_id': pd.NA,
                    'batch_mapping_basis': 'Replicate block + article Methods 2.3',
                    'assay_id': assay, 'metric_id': metric, 'value': float(value),
                    'unit': metric_units[metric], 'observation_level': level,
                    'within_batch_position': position, 'original_local_label': str(original_label) if pd.notna(original_label) else pd.NA,
                    'actual_specimen_id': pd.NA, 'n_items_in_group_reported': n_group_items,
                    'n_expected_values_per_batch': expected_values,
                    'source_file': 'BiscuitData.xlsx', 'source_sheet': sheet,
                    'source_cell': address,
                    'source_label_cell': cell_name(row, start_col - 2 if level == 'technical_repeat' else start_col - 1) if pd.notna(original_label) else pd.NA,
                    'source_metric_header': original_header,
                    'source_formula': formula,
                    'value_origin': 'excel_formula_cache' if formula else 'published_numeric_cell',
                    'qc_flags': qc,
                })

for metric, row in [('weight_before_g', 1), ('weight_after_g', 2), ('weight_loss_pct', 3)]:
    add_block('Table 3_WL', metric, [row], [1, 8, 15], 0, row, 'baking_weight', 'batch_group', 1, 10)
for sheet, metric in [('Table 3_aw', 'water_activity'), ('Table 3_Moisture', 'moisture_pct')]:
    add_block(sheet, metric, range(1, 4), [2, 10, 18], 0, 1, metric, 'technical_repeat', 3, 10)
for metric, row in [('height_mm', 1), ('width_mm', 2), ('length_mm', 3)]:
    add_block('Table 4_Dimensions', metric, [row], [1, 8, 15], 0, row, 'dimensions', 'batch_group', 1, 10, 'dimensions_documentation_cm_vs_header_mm')
for sheet, metric, assay in [
    ('Figure 3_Resistance to penetrat', 'penetration_resistance_ns', 'penetration'),
    ('Figure 3_Number of peaks', 'number_of_peaks', 'penetration'),
    ('Figure 4_Breaking strength', 'breaking_strength_n', 'bending'),
    ('Figure 4_Fracturability', 'deformation_at_break_mm', 'bending'),
]:
    add_block(sheet, metric, range(2, 12), [1, 8, 15], 1, 1, assay, 'item', 10)
for metric, rows in [('colour_l_star', range(1, 4)), ('colour_a_star', range(4, 7)), ('colour_b_star', range(7, 10))]:
    add_block('Table 5_Colour parameters', metric, rows, [1, 8, 15], 0, min(rows), 'colour', 'item', 3)

physical_measurements = pd.DataFrame(records)
physical_measurements['n_items_in_group_reported'] = physical_measurements['n_items_in_group_reported'].astype('Int64')
local_key = ['recipe_id', 'batch_id', 'metric_id', 'original_local_label']
physical_measurements['duplicate_local_label'] = (
    physical_measurements['original_local_label'].notna()
    & physical_measurements.duplicated(local_key, keep=False)
)
physical_measurements.loc[physical_measurements['duplicate_local_label'], 'qc_flags'] = (
    physical_measurements.loc[physical_measurements['duplicate_local_label'], 'qc_flags']
    .map(lambda value: ';'.join(filter(None, [value, 'local_label_repeated_keep_source_rows'])))
)
physical_measurements = physical_measurements.merge(
    recipe_meta[['recipe_id', 'recipe_name_ru', 'recipe_order']], on='recipe_id', validate='many_to_one')
physical_measurements = physical_measurements.merge(
    physical_dictionary[['metric_id', 'metric_name_ru']], on='metric_id', validate='many_to_one')
physical_measurements = physical_measurements.sort_values(
    ['recipe_order', 'source_replicate', 'metric_id', 'within_batch_position'], kind='stable').reset_index(drop=True)

assert physical_measurements['measurement_id'].is_unique
assert len(physical_measurements) == 905
assert physical_measurements['batch_id'].nunique() == 15
assert physical_measurements[['actual_batch_id', 'experiment_date', 'shared_day_id', 'actual_specimen_id']].isna().all().all()
assert physical_measurements.loc[physical_measurements['assay_id'].eq('colour'), 'original_local_label'].isna().all()
assert len(missing_records) == 10  # По одной недостающей ячейке в двух метриках пяти рецептур.
assert physical_measurements.loc[physical_measurements['metric_id'].eq('number_of_peaks'), 'value'].mod(1).eq(0).all()

# Проверяем кэш WL по исходным массам; исходные значения не заменяем расчётом.
weights = physical_measurements.loc[physical_measurements['assay_id'].eq('baking_weight')].pivot(
    index=['recipe_id', 'batch_id'], columns='metric_id', values='value')
wl_recalculated = 100 * (weights['weight_before_g'] - weights['weight_after_g']) / weights['weight_before_g']
assert np.allclose(wl_recalculated, weights['weight_loss_pct'], rtol=1e-12, atol=1e-10)

display(physical_measurements.groupby(['metric_id', 'metric_name_ru', 'unit'], sort=False).size()
        .rename('Доступных значений').reset_index().drop(columns='metric_id').rename(columns={
            'metric_name_ru': 'Показатель', 'unit': 'Единица'}))

,Показатель,Единица,Доступных значений
0,Сила разрушения,Н,145
1,Цветовая координата a*,ед. CIELAB,45
2,Цветовая координата b*,ед. CIELAB,45
3,Светлота L*,ед. CIELAB,45
4,Перемещение при разрушении,мм,145
5,Высота,мм,15
6,Длина,мм,15
7,Влажность,%,45
8,Число пиков,шт.,150
9,Сопротивление проникновению,Н·с,150


### Как сохраняем пропуски и происхождение

В таблицу наблюдений попадают только существующие числовые значения. Десятая строка третьей партии в тесте на излом отсутствует у всех пяти рецептур: пять пропусков силы и пять пропусков перемещения. Нули и искусственные строки не добавляем. В партийной таблице сохраняем ожидаемое и наблюдаемое число значений, а также адреса пустых ячеек.

`measurement_id` — ключ **записи Excel**, построенный из файла, листа и ячейки. `within_batch_position` — позиция строки внутри блока. Ни одно из этих полей не выдаётся за реальный лабораторный ID изделия. Повторяющиеся локальные метки сохраняются; строки с ними не удаляются. Разные разрушающие испытания не соединяются по таким меткам.

В таблицах есть условный `batch_id`, номер исходного блока, показатель, единица, исходная подпись, формула (если была), адрес ячейки и флаги качества. Реальные даты, общие дни и лабораторные ID остаются пустыми.

### Средние по партиям

Внутри партии усредняем изделия или технические повторы; групповой результат оставляем как есть. Для следующего этапа основная статистическая таблица — `physical_batch_means.csv`: одна строка на рецептуру, условную партию и показатель.

Две массы группы нужны для контроля расчёта потерь, поэтому имеют роль `context`. Основная метрика — сила разрушения; ещё 12 показателей качества имеют роль `secondary`. Число пиков и сила не объединяются в общий балл.

In [5]:
def union_flags(values):
    flags = {flag for value in values.dropna() for flag in str(value).split(';') if flag}
    return ';'.join(sorted(flags))

def cell_range(values):
    return ';'.join(values)

batch_keys = ['recipe_id', 'batch_id', 'source_replicate', 'metric_id']
physical_batch_means = physical_measurements.groupby(batch_keys, sort=False).agg(
    batch_mean=('value', 'mean'),
    n_values_observed=('value', 'count'),
    n_values_expected=('n_expected_values_per_batch', 'first'),
    observation_level=('observation_level', 'first'),
    n_items_in_group_reported=('n_items_in_group_reported', 'first'),
    source_file=('source_file', 'first'), source_sheet=('source_sheet', 'first'),
    source_cells=('source_cell', cell_range), qc_flags=('qc_flags', union_flags),
).reset_index()
missing_map = {}
for recipe_id, batch_id, metric, address in missing_records:
    missing_map.setdefault((recipe_id, batch_id, metric), []).append(address)
physical_batch_means['missing_source_cells'] = [
    ';'.join(missing_map.get((r.recipe_id, r.batch_id, r.metric_id), []))
    for r in physical_batch_means.itertuples()
]
physical_batch_means['n_values_missing'] = physical_batch_means['n_values_expected'] - physical_batch_means['n_values_observed']
incomplete = physical_batch_means['n_values_missing'].gt(0)
physical_batch_means.loc[incomplete, 'qc_flags'] = physical_batch_means.loc[incomplete, 'qc_flags'].map(
    lambda value: ';'.join(filter(None, [value, 'one_expected_item_measurement_missing'])))
physical_batch_means['n_independent_batches_in_row'] = 1
physical_batch_means['statistical_unit'] = 'production_batch'
physical_batch_means['actual_batch_id'] = pd.NA
physical_batch_means['experiment_date'] = pd.NA
physical_batch_means['shared_day_id'] = pd.NA
physical_batch_means['aggregation_rule'] = 'mean_within_source_replicate_block'
physical_batch_means = physical_batch_means.merge(recipe_meta, on='recipe_id', validate='many_to_one')
physical_batch_means = physical_batch_means.merge(
    physical_dictionary[['metric_id', 'metric_name_ru', 'unit', 'metric_role']], on='metric_id', validate='many_to_one')
physical_batch_means = physical_batch_means.sort_values(
    ['recipe_order', 'metric_id', 'source_replicate'], kind='stable').reset_index(drop=True)

assert len(physical_batch_means) == 225  # 5 рецептур × 3 партии × 15 показателей.
assert not physical_batch_means.duplicated(['recipe_id', 'batch_id', 'metric_id']).any()
assert physical_batch_means.groupby(['recipe_id', 'metric_id'])['batch_id'].nunique().eq(3).all()
assert physical_batch_means['n_values_missing'].sum() == 10
breaking_counts = physical_batch_means.loc[physical_batch_means['metric_id'].eq('breaking_strength_n')]
assert breaking_counts['n_values_observed'].tolist() == [10, 10, 9] * 5

display(breaking_counts[['recipe_name_ru', 'source_replicate', 'n_values_observed', 'n_values_expected', 'missing_source_cells']]
        .rename(columns={'recipe_name_ru': 'Рецептура', 'source_replicate': 'Условная партия',
                         'n_values_observed': 'Измерений есть', 'n_values_expected': 'По методам',
                         'missing_source_cells': 'Пустая исходная ячейка'}))

,Рецептура,Условная партия,Измерений есть,По методам,Пустая исходная ячейка
0,Стандартная,1,10,10,
1,Стандартная,2,10,10,
2,Стандартная,3,9,10,P12
45,Кукурузный декстрин № 1,1,10,10,
46,Кукурузный декстрин № 1,2,10,10,
47,Кукурузный декстрин № 1,3,9,10,Q12
90,Кукурузный декстрин № 2,1,10,10,
91,Кукурузный декстрин № 2,2,10,10,
92,Кукурузный декстрин № 2,3,9,10,R12
135,Более растворимый инулин,1,10,10,


### Сенсорные средние и русский словарь

Это 40 характеристик, по пять опубликованных средних. Восемь дегустаторов дважды оценивали образцы, но индивидуальных строк нет. Поэтому не рассчитываем новые p-value, стандартные ошибки или интервалы. Опубликованные p-value и буквенные группы сохраняем отдельно как **результаты авторов**, не как результаты нашего анализа. `<.0001` остаётся неравенством, а не нулём и не точным p-value.

Сладость аромата, вкуса и послевкусия — разные признаки. Ключ словаря состоит из раздела и названия. Определения и крайние значения шкал сверяем с S1; для `Dry` при четвёртом укусе определения в S1 нет. Такой пробел явно отмечаем.

In [6]:
section_map = {
    'appearance': ('appearance', 'Внешний вид'),
    'aroma': ('aroma', 'Аромат'),
    'mouthfeelonfirstbite': ('first_bite', 'Первый укус'),
    'mouthfeelonsecondbite': ('second_bite', 'Второй укус'),
    'flavouronthirdbite': ('flavour', 'Третий укус: вкус'),
    'tasteandflavourafterthirdbite': ('flavour', 'Третий укус: вкус'),
    'mouthfeelonfourthbite': ('fourth_bite', 'Четвёртый укус: ощущения при жевании'),
    'aftereffects': ('after_effects', 'Послевкусие и ощущения после проглатывания'),
}

def normalize_label(text):
    return re.sub(r'[^a-z0-9]', '', text.lower())

sensory_terms = {
    'appearance': {
        'surfacecolour': ('Цвет верхней поверхности', 'Степень потемнения верхней поверхности.'),
        'basecolour': ('Цвет основания', 'Степень потемнения нижней поверхности.'),
        'smoothnessofsurface': ('Гладкость поверхности', 'Насколько верхняя поверхность выглядит гладкой, а не шероховатой.'),
        'evennessofcolour': ('Равномерность цвета', 'Насколько окраска изделия однородна.'),
        'smoothnessofsides': ('Гладкость боковых поверхностей', 'Насколько боковые поверхности выглядят гладкими.'),
        'brownspotsincrumb': ('Коричневые пятна внутри', 'Выраженность коричневых участков в середине изделия.'),
    },
    'aroma': {
        'sweet': ('Сладкий аромат', 'Выраженность аромата, напоминающего сахар демерара.'),
        'burnt': ('Подгорелый аромат', 'Выраженность аромата подгоревшей пищи.'),
        'buttery': ('Аромат сливочного масла', 'Выраженность аромата растопленного сливочного масла.'),
        'floury': ('Мучной аромат', 'Выраженность аромата муки или сырого теста.'),
        'cabbage': ('Капустный аромат', 'Выраженность аромата варёной капусты.'),
        'baked': ('Аромат выпечки', 'Аромат, для которого в S1 используется ориентир вафельного рожка.'),
    },
    'first_bite': {
        'firmnessofbite': ('Твёрдость первого укуса', 'Усилие, которое ощущается при первом укусе печенья.'),
    },
    'second_bite': {
        'densityofmouthfeel': ('Плотность во рту', 'Ощущаемая плотность изделия при втором укусе.'),
    },
    'flavour': {
        'sweet': ('Сладость вкуса', 'Интенсивность сладкого вкуса, характерного для сахарозы.'),
        'salty': ('Солёность вкуса', 'Интенсивность солёного вкуса.'),
        'bitter': ('Горечь вкуса', 'Интенсивность горького вкуса.'),
        'burnt': ('Подгорелый вкус', 'Ощущение подгорелой пищи во рту.'),
        'buttery': ('Вкус сливочного масла', 'Ощущение сливочного масла во рту.'),
        'floury': ('Мучной вкус', 'Ощущение муки или сырого теста во вкусе.'),
        'baked': ('Вкус выпечки', 'Ощущение выпечки во рту; ориентир S1 — вафельный рожок.'),
        'metallic': ('Металлический привкус', 'Выраженность металлического привкуса.'),
        'caramel': ('Карамельный вкус', 'Вкус, напоминающий карамельный сироп.'),
    },
    'fourth_bite': {
        'crumbly': ('Крошливость', 'Насколько печенье распадается на мелкие крошки при жевании.'),
        'crunchy': ('Хрусткость', 'Ощущение усилия и характерного громкого хруста при разжёвывании.'),
        'pasting': ('Образование вязкой массы', 'Насколько изделие образует связную массу при жевании.'),
        'dry': ('Сухость при жевании', 'Определение Dry отсутствует в S1; название переведено из Excel без восстановления крайних значений шкалы.'),
        'floury': ('Мучнистость при жевании', 'Ощущение мелкого мучного порошка во рту.'),
        'granular': ('Зернистость', 'Ощущение крупных частиц во рту.'),
        'rateofclearance': ('Скорость очищения рта', 'Как быстро продукт исчезает изо рта при проглатывании.'),
    },
    'after_effects': {
        'sweet': ('Сладкое послевкусие', 'Сладость, ощущаемая после проглатывания.'),
        'salty': ('Солёное послевкусие', 'Солёность, ощущаемая после проглатывания.'),
        'bitter': ('Горькое послевкусие', 'Горечь, ощущаемая после проглатывания.'),
        'burnt': ('Подгорелое послевкусие', 'Подгорелый вкус после проглатывания.'),
        'sour': ('Кислое послевкусие', 'Кислый вкус после проглатывания.'),
        'liquorice': ('Лакричное послевкусие', 'Вкус лакрицы после проглатывания.'),
        'salivating': ('Слюноотделение', 'Выраженность выделения слюны после проглатывания.'),
        'drying': ('Сухость после проглатывания', 'Ощущение сухости во рту после проглатывания.'),
        'numbing': ('Онемение языка', 'Ощущение снижения чувствительности языка после проглатывания.'),
        'toothpacking': ('Налипание на зубы', 'Насколько остатки изделия остаются на поверхности зубов.'),
    },
}
anchor_translation = {
    'Light to Dark': 'От светлого к тёмному', 'Not to Very': 'От отсутствия до сильной выраженности',
    'None to Lots': 'От отсутствия до большого количества', 'Soft to Hard': 'От мягкого к твёрдому',
    'Not (light) to Very (dense)': 'От лёгкого к плотному', 'Slow to Fast': 'От медленного к быстрому',
    'Low to Strong': 'От слабого к сильному',
}

s1_definitions = {}
section_id = None
for row_number, row in enumerate(Document(SOURCE / 'article_supplement_mmc1.docx').tables[0].rows, 1):
    values = [cell.text.strip() for cell in row.cells]
    label_key = normalize_label(values[0])
    if label_key in section_map:
        section_id = section_map[label_key][0]
    elif section_id is not None and len(values) == 3 and values[1]:
        key = (section_id, label_key)
        assert key not in s1_definitions
        s1_definitions[key] = {'definition_original': values[1], 'anchors_original': values[2], 's1_row': row_number}
assert len(s1_definitions) == 39

sensory_rows = []
sensory_dictionary_rows = []
sheet_name = 'Table 6_sensory profile of bisc'
frame = biscuit_sheets[sheet_name]
section_id = section_ru = section_original = None
for row in range(1, len(frame)):
    label = frame.iat[row, 0]
    if pd.isna(label):
        continue
    if pd.isna(frame.iat[row, 1]):
        section_id, section_ru = section_map[normalize_label(label)]
        section_original = label
        continue
    attribute_key = normalize_label(label)
    term_ru, definition_ru = sensory_terms[section_id][attribute_key]
    metric_id = f'sensory_{section_id}_{attribute_key}'
    definition = s1_definitions.get((section_id, attribute_key), {})
    definition_status = 'supported_by_S1' if definition else 'definition_missing_in_S1'
    sensory_dictionary_rows.append({
        'metric_id': metric_id, 'metric_name_ru': term_ru, 'metric_original': label,
        'domain': 'sensory', 'section_ru': section_ru, 'section_original': section_original,
        'metric_label_ru': f'{section_ru} · {term_ru}', 'unit': 'баллы 0–100', 'delta_unit': 'пункты шкалы',
        'definition_ru': definition_ru, 'metric_role': 'secondary', 'relative_delta_allowed': False,
        'statistical_unit': 'unavailable_individual_sensory_ratings', 'new_statistical_tests_allowed': False,
        'definition_source_file': 'article_supplement_mmc1.docx' if definition else 'BiscuitData.xlsx',
        'definition_source_location': f"S1, row {definition['s1_row']}" if definition else f'{sheet_name}!A{row + 1}',
        'definition_status': definition_status,
        'definition_original': definition.get('definition_original', pd.NA),
        'anchors_original': definition.get('anchors_original', pd.NA),
        'anchors_ru': anchor_translation.get(definition.get('anchors_original'), pd.NA),
    })
    p_raw = frame.iat[row, 11]
    if isinstance(p_raw, str) and p_raw.startswith('<'):
        p_operator, p_bound, p_exact = '<', float(p_raw[1:]), np.nan
    else:
        p_operator, p_bound, p_exact = '=', np.nan, float(p_raw)
    for original, col in zip(recipe_original_order, [1, 3, 5, 7, 9]):
        assert frame.iat[0, col] == original
        value = float(frame.iat[row, col])
        sensory_rows.append({
            'recipe_id': original_to_id[original], 'metric_id': metric_id,
            'section_original': section_original, 'attribute_original': label,
            'published_mean': value, 'unit': 'баллы 0–100',
            'data_status': 'published_summary_only', 'raw_individual_ratings_available': False,
            'new_statistical_tests_allowed': False,
            'n_panelists_reported': 8, 'n_scoring_sessions_reported': 2,
            'n_independent_batches': pd.NA,
            'published_p_text': str(p_raw), 'published_p_operator': p_operator,
            'published_p_exact': p_exact, 'published_p_upper_bound': p_bound,
            'published_p_scope': 'overall_sample_effect_from_authors',
            'published_letters': frame.iat[row, col + 1],
            'source_file': 'BiscuitData.xlsx', 'source_sheet': sheet_name,
            'source_cell': cell_name(row, col), 'source_p_cell': cell_name(row, 11),
            'source_letters_cell': cell_name(row, col + 1),
            'definition_status': definition_status,
            'qc_flags': 'individual_ratings_not_public' + (';definition_missing_in_S1' if not definition else ''),
        })

sensory_dictionary = pd.DataFrame(sensory_dictionary_rows)
metric_dictionary = pd.concat([physical_dictionary, sensory_dictionary], ignore_index=True)
assert metric_dictionary['metric_id'].is_unique
assert len(metric_dictionary) == 55

sensory_means = pd.DataFrame(sensory_rows).merge(recipe_meta, on='recipe_id', validate='many_to_one')
sensory_means = sensory_means.merge(
    sensory_dictionary[['metric_id', 'metric_name_ru', 'metric_label_ru', 'section_ru']], on='metric_id', validate='many_to_one')
sensory_control = sensory_means.loc[sensory_means['recipe_id'].eq('control'), ['metric_id', 'published_mean']].rename(
    columns={'published_mean': 'control_published_mean'})
sensory_means = sensory_means.merge(sensory_control, on='metric_id', validate='many_to_one')
sensory_means['delta_vs_control_points'] = sensory_means['published_mean'] - sensory_means['control_published_mean']
sensory_means = sensory_means.sort_values(['recipe_order', 'source_cell'], kind='stable').reset_index(drop=True)
assert len(sensory_means) == 200
assert not sensory_means.duplicated(['recipe_id', 'metric_id']).any()
assert sensory_means['published_mean'].between(0, 100).all()
assert sensory_means['n_independent_batches'].isna().all()
assert sensory_means['definition_status'].eq('definition_missing_in_S1').sum() == 5
assert sensory_means.loc[sensory_means['published_p_operator'].eq('<'), 'published_p_exact'].isna().all()
sensory_definition_view = sensory_dictionary[['section_ru', 'metric_name_ru', 'definition_status']].copy()
sensory_definition_view['definition_status'] = sensory_definition_view['definition_status'].replace({
    'supported_by_S1': 'Есть определение в S1', 'definition_missing_in_S1': 'Нет определения в S1'})
display(sensory_definition_view.rename(columns={
    'section_ru': 'Раздел', 'metric_name_ru': 'Показатель', 'definition_status': 'Статус определения'}))

,Раздел,Показатель,Статус определения
0,Внешний вид,Цвет верхней поверхности,Есть определение в S1
1,Внешний вид,Цвет основания,Есть определение в S1
2,Внешний вид,Гладкость поверхности,Есть определение в S1
3,Внешний вид,Равномерность цвета,Есть определение в S1
4,Внешний вид,Гладкость боковых поверхностей,Есть определение в S1
5,Внешний вид,Коричневые пятна внутри,Есть определение в S1
6,Аромат,Сладкий аромат,Есть определение в S1
7,Аромат,Подгорелый аромат,Есть определение в S1
8,Аромат,Аромат сливочного масла,Есть определение в S1
9,Аромат,Мучной аромат,Есть определение в S1


### Какие данные пока остаются вне аналитических витрин

В `DoughData.xlsx` есть шесть серий на рецептуру, но нет их связи с тремя партиями. Для шестых серий Orafti и Fibruline не опубликован G″; некоторые готовые средние модулей не совпадают с доступными исходными сериями. В книге есть внешние ссылки и 38 формул `#REF!`. Частотные точки одного образца зависимы. Эти данные прочитаны для проверки, но не включаются в текущие статистические и BI-витрины.

В `IngredientData.xlsx` происхождение трёх HPLC-повторов не уточнено. Формулы сумм Nutriose и Promitor пропускают часть представленных пиков, готовые проценты смешивают формулы и константы. Нормализованные DP-профили по сомнительным знаменателям не создаём.

Это не удаление исходных данных: оригиналы и подробный аудит сохраняются. Отдельные SD из статьи также не копируем в наши расчёты — уровень наблюдения и формулы отличаются или не воспроизводятся.

In [7]:
# Повторно фиксируем проблемы исходных книг, не исправляя их.
dough_formulas = load_workbook(SOURCE / 'DoughData.xlsx', data_only=False, read_only=True)
ref_errors = [(sheet.title, cell.coordinate) for sheet in dough_formulas for row in sheet for cell in row
              if isinstance(cell.value, str) and '#REF!' in cell.value]
assert len(ref_errors) == 38
dp = books['IngredientData.xlsx']['Table 1_DP']
dp_checks = []
for original, starts in [('Nutriose', [2, 8, 14]), ('Promitor', [3, 9, 15])]:
    for replicate, col in enumerate(starts, 1):
        present_peak_sum = pd.to_numeric(dp.iloc[2:8, col], errors='raise').sum(min_count=1)
        source_total = float(dp.iat[8, col])
        assert not np.isclose(present_peak_sum, source_total)
        dp_checks.append((original, replicate, present_peak_sum, source_total))
display(pd.DataFrame(dp_checks, columns=['Ингредиент авторов', 'Повтор HPLC', 'Сумма представленных пиков', 'Total в файле']))
dough_formulas.close()
formula_book.close()

,Ингредиент авторов,Повтор HPLC,Сумма представленных пиков,Total в файле
0,Nutriose,1,1744636.0,1738950.0
1,Nutriose,2,1724760.0,1719059.0
2,Nutriose,3,1732610.0,1727034.0
3,Promitor,1,1314607.0,1203902.0
4,Promitor,2,1417192.0,1309554.0
5,Promitor,3,1396156.0,1284502.0


## 3. Сколько сахарозы заменили и что произошло с качеством

**На этом этапе — подготовка числовой витрины и план сравнения.** Итоговые графики и выводы о качестве ещё не строим.

Среднее рецептуры рассчитываем как среднее трёх партийных средних. Каждая партия имеет одинаковый вес; наличие 9 вместо 10 изделий в третьей партии не уменьшает вес этой партии. Сохраняем описательное выборочное SD между партиями (`ddof=1`), минимум, максимум и число партий. SD показывает разброс, а не доверительный интервал.

Отклонение — среднее варианта минус среднее стандарта. Для силы, размеров и других показателей с осмысленной относительной шкалой дополнительно вычисляем процент от контрольного среднего. Для цветовых координат и aw относительное отклонение не создаём. Если контрольное среднее равно нулю, процент также остаётся пустым. Разность влажности или потерь массы выражаем в **процентных пунктах**, а не в процентах относительного изменения.

План раздела после согласования: сопоставить снижение сахарозы и изменение силы разрушения, затем проверить остальные характеристики. Не строить регрессию «процент снижения → качество» по пяти рецептурам с разными ингредиентами и дозировками. Общий балл с произвольными весами и технологические допуски не вводить.

In [8]:
physical_recipe_summary = physical_batch_means.groupby(['recipe_id', 'metric_id'], sort=False).agg(
    mean_of_batch_means=('batch_mean', 'mean'),
    sd_between_batches=('batch_mean', 'std'),
    min_batch_mean=('batch_mean', 'min'), max_batch_mean=('batch_mean', 'max'),
    n_independent_batches=('batch_id', 'nunique'),
    n_values_observed_total=('n_values_observed', 'sum'),
    n_values_expected_total=('n_values_expected', 'sum'),
    n_values_missing_total=('n_values_missing', 'sum'),
    observation_level=('observation_level', 'first'),
    source_file=('source_file', 'first'), source_sheet=('source_sheet', 'first'),
    source_cells=('source_cells', cell_range), qc_flags=('qc_flags', union_flags),
).reset_index()
physical_recipe_summary = physical_recipe_summary.merge(
    physical_dictionary[['metric_id', 'metric_name_ru', 'unit', 'delta_unit', 'metric_role', 'relative_delta_allowed']],
    on='metric_id', validate='many_to_one')
physical_recipe_summary = physical_recipe_summary.merge(
    recipes[['recipe_id', 'recipe_name_ru', 'recipe_order', 'recipe_original',
             'sucrose_g_per_100g_flour', 'fibre_ingredient_g_per_100g_flour', 'sucrose_reduction_pct', 'qc_flags']]
    .rename(columns={'qc_flags': 'recipe_qc_flags'}), on='recipe_id', validate='many_to_one')
control_means = physical_recipe_summary.loc[physical_recipe_summary['recipe_id'].eq('control'), ['metric_id', 'mean_of_batch_means']].rename(
    columns={'mean_of_batch_means': 'control_mean'})
physical_recipe_summary = physical_recipe_summary.merge(control_means, on='metric_id', validate='many_to_one')
physical_recipe_summary['delta_vs_control'] = physical_recipe_summary['mean_of_batch_means'] - physical_recipe_summary['control_mean']
physical_recipe_summary['relative_delta_pct'] = np.nan
relative_allowed = physical_recipe_summary['relative_delta_allowed'] & physical_recipe_summary['control_mean'].ne(0)
physical_recipe_summary.loc[relative_allowed, 'relative_delta_pct'] = (
    100 * physical_recipe_summary.loc[relative_allowed, 'delta_vs_control'] / physical_recipe_summary.loc[relative_allowed, 'control_mean'])
physical_recipe_summary['mean_weighting'] = 'equal_weight_per_production_batch'
physical_recipe_summary['statistical_unit'] = 'production_batch'
physical_recipe_summary['data_status'] = 'descriptive_summary_no_inference'
physical_recipe_summary['uncertainty_available'] = 'between_batch_SD_only_not_CI'
physical_recipe_summary['day_pairing_confirmed'] = False
physical_recipe_summary = physical_recipe_summary.sort_values(['recipe_order', 'metric_id'], kind='stable').reset_index(drop=True)

assert len(physical_recipe_summary) == 75
assert physical_recipe_summary['n_independent_batches'].eq(3).all()
assert physical_recipe_summary.loc[physical_recipe_summary['recipe_id'].eq('control'), 'delta_vs_control'].eq(0).all()
assert physical_recipe_summary.loc[~physical_recipe_summary['relative_delta_allowed'], 'relative_delta_pct'].isna().all()

# Целостность каждой исходной ячейки и расчёта партийного среднего.
for record in physical_measurements.itertuples():
    row_number = int(re.search(r'[0-9]+$', record.source_cell).group()) - 1
    column_letters = re.match(r'[A-Z]+', record.source_cell).group()
    col_number = column_index_from_string(column_letters) - 1
    assert record.value == float(source_value(record.source_sheet, row_number, col_number))
expected_batch_means = physical_measurements.groupby(batch_keys)['value'].mean().sort_index()
actual_batch_means = physical_batch_means.set_index(batch_keys)['batch_mean'].sort_index()
pd.testing.assert_series_equal(expected_batch_means, actual_batch_means, check_names=False)
assert metric_dictionary['metric_id'].isin(set(physical_measurements['metric_id']) | set(sensory_means['metric_id'])).all()

prepared_tables = {
    'recipes.csv': recipes,
    'physical_measurements.csv': physical_measurements,
    'physical_batch_means.csv': physical_batch_means,
    'sensory_means.csv': sensory_means,
    'metric_dictionary.csv': metric_dictionary,
    'physical_recipe_summary.csv': physical_recipe_summary,
}
for filename, frame in prepared_tables.items():
    frame.to_csv(DATA / filename, index=False, encoding='utf-8-sig')
for filename, original_digest in source_hashes_before.items():
    assert hashlib.sha256((SOURCE / filename).read_bytes()).hexdigest() == original_digest

display(pd.DataFrame([(name, len(frame), len(frame.columns)) for name, frame in prepared_tables.items()],
                     columns=['Подготовленный файл', 'Строк', 'Столбцов']))
print('Исходники сохранены без изменения. Новые тесты и доверительные интервалы не рассчитывались.')

,Подготовленный файл,Строк,Столбцов
0,recipes.csv,5,31
1,physical_measurements.csv,905,30
2,physical_batch_means.csv,225,29
3,sensory_means.csv,200,35
4,metric_dictionary.csv,55,20
5,physical_recipe_summary.csv,75,35


Исходники сохранены без изменения. Новые тесты и доверительные интервалы не рассчитывались.


### Точная структура статистических и BI-таблиц

Файлы CSV — UTF-8 с BOM, разделитель запятая, десятичная точка. Пропуск — пустое поле, не ноль. Точность хранится для расчётов, округление до 2 знаков используется только в пользовательском отображении.

| Файл | Уровень строки / ключ | Для чего используется |
|---|---|---|
| `recipes.csv` | Одна рецептура / `recipe_id` | Полный состав из таблицы 2, дозировка ингредиента с волокнами, вычисленное снижение сахарозы, общий вес и флаг Orafti; графики состава |
| `physical_measurements.csv` | Одно существующее значение / `measurement_id` | Проверка ячейки, единиц, формулы и вложенности; `batch_id` — условный, `actual_specimen_id` пуст |
| `physical_batch_means.csv` | Рецептура × партия × показатель / `recipe_id, batch_id, metric_id` | **Основной вход будущих статистических тестов**; `batch_mean`, наблюдаемое/ожидаемое число измерений, адреса пропусков |
| `physical_recipe_summary.csv` | Рецептура × показатель / `recipe_id, metric_id` | **Физическая BI-витрина**: `mean_of_batch_means`, `control_mean`, `delta_vs_control`, `relative_delta_pct`, `unit`, `delta_unit`, `sd_between_batches`, `n_independent_batches`, состав и ограничения |
| `sensory_means.csv` | Рецептура × сенсорный признак / `recipe_id, metric_id` | **Отдельная сенсорная BI-витрина**: `published_mean`, контрольное среднее, разность в пунктах; статус сводных данных; число независимых партий неизвестно |
| `metric_dictionary.csv` | Один показатель / `metric_id` | Русские названия, определения, единицы, разделы, источник определения, роль основной/дополнительной метрики и разрешение новых тестов |

Связи: каждая таблица фактов соединяется с `recipes` по `recipe_id` и со словарём по `metric_id` как «многие к одному». Физические и сенсорные факты напрямую не соединяем. Сопоставление двух рецептур в BI использует уже рассчитанное Python поле контроля, не объединяет партии одинакового номера. Не суммируем `n_independent_batches` по нескольким показателям: в каждой строке витрины оно относится к одному конкретному показателю.

Для будущих результатов тестов планируется одна таблица с ключом `metric_id, recipe_id, control_recipe_id, method_id`: эффект в исходной единице, границы интервала, тип и уровень интервала, p-value и поправка, число партий и описание модели. Сейчас её не создаём: нет утверждённого метода и результатов. Интервалы впоследствии соединяются с физической витриной по ключам показателя и рецептуры, а не копируются из статьи.

Для ΔE*76 будущий шаг: сначала получить средние L*, a*, b* каждой рецептуры с равными весами партий, затем вычислить расстояние до контрольного цветового центра. Контрольные и экспериментальные партии не спариваются. Расстояние между средними цветами — описательный показатель; порог производственной приемлемости не задаётся. Сейчас сохраняем исходные координаты, чтобы не вводить ещё одну метрику до согласования.

### План Yandex DataLens

Название: **«Как снижение сахарозы влияет на качество печенья»**.

Два отдельных блока — физические характеристики и опубликованная сенсорика. Фильтры: рецептура, физический показатель; для сенсорики — раздел и признак. Порядок рецептур задаётся `recipe_order`, названия — русские. Стандарт всегда остаётся базой сравнения, даже при выборе одного варианта.

| Элемент | Данные и поля | Правило отображения |
|---|---|---|
| 1. Содержание добавленной сахарозы | `recipes`: рецепт, `sucrose_g_per_100g_flour` | Г/100 г муки; это не сахар готового продукта |
| 2. Снижение сахарозы | `recipes`: `sucrose_reduction_pct` | Фактические проценты из состава; флаг расхождения с заявленными 30% |
| 3. Сила разрушения | `physical_recipe_summary`, фильтр `breaking_strength_n`; точки из `physical_batch_means` | Среднее трёх партий и значения партий. SD можно показывать только с подписью «разброс между партиями»; CI — после согласования модели |
| 4. Отличия выбранного физического показателя | Физическая витрина: среднее, контроль, абсолютное/разрешённое относительное отклонение | Один показатель и одна единица на шкале. Для влажности/потерь разность в п.п.; для цвета — исходная координата |
| 5. Сенсорные характеристики | `sensory_means`: русское имя, `published_mean`, `delta_vs_control_points` | Опубликованные средние, без придуманных интервалов и новых тестов; не оценка потребительского предпочтения |
| Итоговая таблица | `recipes` + отдельные отфильтрованные показатели физических/сенсорных витрин | Пять рецептур, снижение сахарозы, ключевые показатели, единицы, число партий, пропуски и ограничения. Каждая метрика — отдельный столбец |

Карточка выбранного физического показателя: среднее варианта, контрольное среднее, разность, единица, 3 независимые партии и число доступных вложенных измерений. Для сенсорики вместо «3 партии» — «опубликованные средние; производственная привязка неизвестна».

На дашборде будут видны ограничения: только исследованные рецептуры; нет измерения всех сахаров готового изделия; три партии; не подтверждена парность дней; нет индивидуальной сенсорики; Orafti имеет расхождение общей массы. Физические массы группы с ролью `context` не входят в список основных характеристик качества.

## 4. Статистическая проверка

**План следующего этапа; тестов здесь нет.** Основная предварительная метрика — сила разрушения в Н. Сначала показать три партийных значения каждой рецептуры и оценить допущения, затем утвердить метод четырёх сравнений со стандартом.

1. Уточнить, были ли общие дни для всех рецептур. Нельзя создавать парность из номеров `Replicate`.
2. При обоснованной модели независимых партий и подходящей структуре ошибок рассмотреть ANOVA и сравнения Dunnett с контролем. При неодинаковых дисперсиях — варианты Welch/контрастов с поправкой на множественность. Это кандидаты, не уже выбранные методы.
3. Если общие дни подтвердятся, рассмотреть день как блок. При полном дизайне 5×3 отдельное взаимодействие рецептуры с днём без дополнительных партий не оценивается.
4. Сравнивать партийные средние, сообщать эффект в Н и, при обоснованной модели, интервалы контрастов. Для дополнительных показателей определить семейства сравнений и поправки. Силу разрушения и сенсорную твёрдость укуса не считать одним измерением.
5. Указать малую выборку: n=3 партии на рецептуру; проверки нормальности не подтверждают нормальность при таком n. Перестановки должны сохранять уровень партии и иметь обоснованную обменяемость. Bootstrap отдельных печений не допускается.

Отсутствие значимого различия не означает эквивалентность. Для её проверки нужны согласованные практические границы и достаточная мощность; границы не придумываем. Тесты по 200 сенсорным средним, шести неподписанным реологическим сериям или частотным точкам не выполняем.

## 5. Выводы и рекомендации производству

**На этапе 2 можно подтвердить готовность данных, но ещё нельзя выбрать лучшую рецептуру.** Все пять составов проверены по статье; физические значения имеют происхождение до ячейки, а таблица будущих тестов сохраняет три партии на рецептуру. Пропуски не заполнены. Сенсорные средние отделены от экспериментальных строк.

План финальных выводов: оценить компромисс между сокращением добавленной сахарозы и близостью силы разрушения к стандарту, проверить дополнительные свойства и учесть опубликованную сенсорику. Предлагать только кандидата для испытаний, а не готовое внедрение. Универсальный предел снижения сахарозы, экономический эффект, потребительскую приемлемость и технологическую эквивалентность этот набор не доказывает.

### Оставшиеся вопросы

- Нужны подтверждённые навески/состав Orafti и объяснение заявленных 30%.
- Нужны даты/ID партий и подтверждение общих экспериментальных дней, прежде чем выбирать блокированный или независимый дизайн.
- Неизвестна причина отсутствия десятых измерений излома в третьей партии. Девять существующих значений сохраняются.
- Для самостоятельной сенсорной проверки нужна индивидуальная выгрузка Compusense с дегустаторами, сессиями, порядком подачи и привязкой к партии, если она фиксировалась. Автору запрос не отправлялся.
- Для возможного включения реологии/HPLC нужны производственная привязка серий, недостающие G″ и объяснение расчётов.

Следующий этап начинается после согласования подготовки данных и логики исследования. Материалы первого аудита и оригинальные экспериментальные файлы сохранены.